In [9]:
import pandas as pd
import numpy as np
import sys
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

sys.path.append('../../../utils')
from SubmissionHelper import save_submission

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

def feature_engineering(df):
    df_copy = df.copy()
    
    # 1. 蒸发压力指数 (温度高、风大、湿度低 = 极度缺水)
    # 加 1e-5 是为了防止分母为 0
    df_copy['Evap_Index'] = (df_copy['Temperature_C'] * df_copy['Wind_Speed_kmh']) / (df_copy['Humidity'] + 1e-5)
    
    # 2. 土壤储水压力
    # 降雨量少且土壤水分低
    df_copy['Water_Stress'] = df_copy['Rainfall_mm'] / (df_copy['Soil_Moisture'] + 1e-5)
    
    # 3. 组合特征 (Object 类型的强强联手)
    # CatBoost 擅长处理这种组合
    df_copy['Crop_Soil'] = df_copy['Crop_Type'].astype(str) + "_" + df_copy['Soil_Type'].astype(str)
    
    # 4. 区域干旱度
    # 统计每个 Region 的平均降雨量（注意：测试集要用训练集的统计值，这里简单处理）
    region_rain = df_copy.groupby('Region')['Rainfall_mm'].transform('mean')
    df_copy['Region_Rain_Diff'] = df_copy['Rainfall_mm'] - region_rain
    
    return df_copy

In [2]:
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
inv_target_map = {0: 'Low', 1: 'Medium', 2: 'High'}
y = train['Irrigation_Need'].map(target_map)

# 确定特征列
X = train.drop(['id', 'Irrigation_Need'], axis=1)
X_test = test.drop(['id'], axis=1)

# 自动识别所有 object 类型的列名
cat_features = list(X.select_dtypes(include=['object']).columns)
print(f"Auto processing: {cat_features}")

Auto processing: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


In [14]:
X_train_fe = feature_engineering(X_train)
X_val_fe = feature_engineering(X_val)
X_test_fe = feature_engineering(X_test)
X_all_fe = feature_engineering(X)
X_test_fe = feature_engineering(X_test)

# 更新分类特征列表（因为新增了 Crop_Soil）
cat_features_fe = list(X_train_fe.select_dtypes(include=['object']).columns)

In [15]:
# --- 1. 先进行全量特征工程 ---
# 注意：要把原始 train 的特征传进去，包含我们要处理的 X 和 X_test
X_all_fe = feature_engineering(X)
X_test_fe = feature_engineering(X_test)

# --- 2. 重新识别分类特征 (必须包含新生成的 Crop_Soil) ---
cat_features_fe = list(X_all_fe.select_dtypes(include=['object']).columns)

# --- 3. 划分数据 (使用处理后的 X_all_fe) ---
X_train_fe, X_val_fe, y_train, y_val = train_test_split(
    X_all_fe, y, test_size=0.2, random_state=42, stratify=y
)

# --- 4. 初始化模型 (参数传入 cat_features_fe) ---
model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=6,
    cat_features=cat_features_fe, # 这里必须用更新后的列表
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100,
    task_type='GPU'
)

# --- 5. 训练 (使用 _fe 结尾的变量) ---
model.fit(
    X_train_fe, y_train,
    eval_set=(X_val_fe, y_val),
    early_stopping_rounds=100,
    use_best_model=True
)

# --- 6. 预测 (使用 X_test_fe) ---
test_preds_num = model.predict(X_test_fe)

0:	learn: 0.9815000	test: 0.9812302	best: 0.9812302 (0)	total: 13.6ms	remaining: 1m 7s
100:	learn: 0.9826766	test: 0.9823889	best: 0.9823889 (99)	total: 1.32s	remaining: 1m 4s
200:	learn: 0.9834385	test: 0.9829762	best: 0.9830159 (193)	total: 2.56s	remaining: 1m 1s
300:	learn: 0.9835456	test: 0.9831746	best: 0.9831746 (300)	total: 3.74s	remaining: 58.4s
400:	learn: 0.9837321	test: 0.9832937	best: 0.9833095 (399)	total: 4.9s	remaining: 56.3s
500:	learn: 0.9837540	test: 0.9833413	best: 0.9833651 (471)	total: 6.04s	remaining: 54.3s
600:	learn: 0.9838591	test: 0.9833571	best: 0.9834048 (568)	total: 7.2s	remaining: 52.7s
700:	learn: 0.9839286	test: 0.9833889	best: 0.9834127 (664)	total: 8.4s	remaining: 51.5s
800:	learn: 0.9839841	test: 0.9833810	best: 0.9834286 (747)	total: 9.58s	remaining: 50.2s
bestTest = 0.9834285714
bestIteration = 747
Shrink model to first 748 iterations.


In [17]:
# 压平(only catboost)
final_preds = test_preds_num.flatten()
# 转换回来
final_labels = [inv_target_map[p] for p in final_preds]

save_submission(final_labels, test, 'id', 'Irrigation_Need', prefix='catboost_fe')

------------------------------
✅ [SUCCESS] Submission file generated!
📍 Location: D:\Kaggle-Learning\02-Tabular-Data\playground-series-s6e4\submissions\catboost_fe_0404_1836.csv
📊 Shape: (270000, 2)
------------------------------


'../submissions\\catboost_fe_0404_1836.csv'

In [18]:
hello='hello'